In [8]:
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import os
import re
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [2]:
BASE = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune"
GSEA_OUT_DIR = f"{BASE}/results/10_integration_GBM_HIHA/10_07_GSEA_perct"
os.makedirs(GSEA_OUT_DIR, exist_ok=True)

deg_df = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/10_integration_GBM_HIHA/10_06_deg_macro_percelltype.csv")

In [3]:
OUT_DIR = f"{BASE}/results/10_integration_GBM_HIHA"

In [4]:
GENE_SETS = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/c5.go.bp.v2026.1.Hs.symbols.gmt"


In [9]:
prerank_results = {}
combined = []
 
for ct in deg_df["celltype"].unique():
    sub = deg_df[deg_df["celltype"] == ct].copy()
 
    rnk = sub[["names", "scores"]].sort_values("scores", ascending=False)
    rnk.columns = ["gene_name", "Wilcoxon score"]
    rnk = rnk.drop_duplicates("gene_name").set_index("gene_name")
 
    print(f"\n[{ct}] running prerank on {len(rnk)} genes")
 
    ct_slug = ct.replace(" ", "_").replace(".", "").replace("/", "_")
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=GENE_SETS,
        min_size=5,
        max_size=1000,
        permutation_num=1000,
        outdir=f"{GSEA_OUT_DIR}/{ct_slug}",
        seed=42,
        threads=4,
    )
    prerank_results[ct] = pre_res
 
    res = pre_res.res2d.copy()
    res.insert(0, "celltype", ct)
    combined.append(res)

2026-09-25 09:14:25,252 [WARNING] Duplicated values found in preranked stats: 28.04% of genes
The order of those genes will be arbitrary, which may produce unexpected results.



[Macro. and mono. prolif.] running prerank on 22586 genes


2026-09-25 09:17:43,301 [WARNING] Duplicated values found in preranked stats: 35.14% of genes
The order of those genes will be arbitrary, which may produce unexpected results.



[TAMs C1QC] running prerank on 22586 genes


2026-09-25 09:21:00,744 [WARNING] Duplicated values found in preranked stats: 35.77% of genes
The order of those genes will be arbitrary, which may produce unexpected results.



[TAMs proinflammatory] running prerank on 22586 genes


2026-09-25 09:24:28,960 [WARNING] Duplicated values found in preranked stats: 29.58% of genes
The order of those genes will be arbitrary, which may produce unexpected results.



[Monocytes] running prerank on 22586 genes


In [10]:
combined_df = pd.concat(combined, ignore_index=True)
out_csv = os.path.join(OUT_DIR, "10_07_gsea_prerank_all_celltypes.csv")
combined_df.to_csv(out_csv, index=False)
print(f"\nSaved combined prerank results to {out_csv}")
 
print("\nTop 5 significant terms per cell type (by FDR q-val):")
sig = combined_df[combined_df["FDR q-val"] < 0.05]
print(
    sig.sort_values(["celltype", "FDR q-val"])
    .groupby("celltype", observed=True)
    .head(5)[["celltype", "Term", "NES", "FDR q-val"]]
    .to_string(index=False)
)


Saved combined prerank results to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/10_integration_GBM_HIHA/10_07_gsea_prerank_all_celltypes.csv

Top 5 significant terms per cell type (by FDR q-val):
                celltype                                                                      Term       NES  FDR q-val
Macro. and mono. prolif.                                              GOBP_CYTOPLASMIC_TRANSLATION -1.792904   0.000000
Macro. and mono. prolif. GOBP_NEGATIVE_REGULATION_OF_SYNCYTIUM_FORMATION_BY_PLASMA_MEMBRANE_FUSION -1.790790   0.000000
Macro. and mono. prolif.                                                      GOBP_MYOBLAST_FUSION -1.743480   0.000000
Macro. and mono. prolif.          GOBP_REGULATION_OF_SYNCYTIUM_FORMATION_BY_PLASMA_MEMBRANE_FUSION -1.738865   0.000000
Macro. and mono. prolif.                                                  GOBP_SYNCYTIUM_FORMATION -1.692513   0.000000
               Monocytes GOBP_NEGATIVE_REGULATION_OF_SYNCYTIUM_F

Dotplot of top GSEA pathways across the 5 macrophage/monocyte cell types.
 
    y-axis: pathway terms (top hits, selected per cell type)
    x-axis: cell type
    dot size: number of leading-edge genes driving that enrichment
    dot color: NES - red = up in GBM (positive), green = up in Healthy (negative)
 
Reads gsea_prerank_all_celltypes.csv from 12_gsea_prerank_percelltype.py.

In [13]:
FDR_CUTOFF = 0.05
TOP_N_PER_CELLTYPE = 5  # top pathways per cell type (by FDR q-val) that feed the combined term list
 
combined_df = pd.read_csv(os.path.join(OUT_DIR, "10_07_gsea_prerank_all_celltypes.csv"))
print("Loaded:", combined_df.shape)
print(combined_df.head)

Loaded: (28956, 11)
<bound method NDFrame.head of                        celltype     Name  \
0      Macro. and mono. prolif.  prerank   
1      Macro. and mono. prolif.  prerank   
2      Macro. and mono. prolif.  prerank   
3      Macro. and mono. prolif.  prerank   
4      Macro. and mono. prolif.  prerank   
...                         ...      ...   
28951                 Monocytes  prerank   
28952                 Monocytes  prerank   
28953                 Monocytes  prerank   
28954                 Monocytes  prerank   
28955                 Monocytes  prerank   

                                                    Term        ES       NES  \
0             GOBP_PROTEIN_CONTAINING_COMPLEX_REMODELING  0.838213  2.031290   
1           GOBP_REGULATION_OF_ASTROCYTE_DIFFERENTIATION  0.874077  1.971051   
2      GOBP_NEGATIVE_REGULATION_OF_GLIAL_CELL_DIFFERE...  0.865567  1.923373   
3                                    GOBP_VASCULOGENESIS  0.698765  1.916482   
4                    

In [12]:
combined_df["n_lead_genes"] = (
    combined_df["Lead_genes"].fillna("").apply(lambda s: len(s.split(";")) if s else 0)
)
sig = combined_df[combined_df["FDR q-val"] < FDR_CUTOFF].copy()
 
top_terms = (
    sig.sort_values("FDR q-val")
    .groupby("celltype", observed=True)
    .head(TOP_N_PER_CELLTYPE)["Term"]
    .unique()
)
print(f"{len(top_terms)} unique pathways selected across all cell types "
      f"(top {TOP_N_PER_CELLTYPE} per cell type, FDR<{FDR_CUTOFF})")
 
plot_df = combined_df[combined_df["Term"].isin(top_terms)].copy()

13 unique pathways selected across all cell types (top 5 per cell type, FDR<0.05)


In [14]:
term_order = (
    plot_df.groupby("Term")["NES"].apply(lambda x: x.abs().max())
    .sort_values(ascending=False).index.tolist()
)
celltype_order = sorted(plot_df["celltype"].unique())
 
plot_df["Term"] = pd.Categorical(plot_df["Term"], categories=term_order[::-1], ordered=True)
plot_df["celltype"] = pd.Categorical(plot_df["celltype"], categories=celltype_order, ordered=True)


In [28]:
cmap = mcolors.LinearSegmentedColormap.from_list(
    "healthy_gbm_diverging", ["#CA425F", "#FFFFFF", "#69DD82"]  #     "Healthy": "#69DD82", "GBM": "#CA425F"
)
vmax = plot_df["NES"].abs().max()
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
 
# ---- 4. only draw a dot where that (celltype, term) combo was actually significant - blank cells stay blank rather than showing a non-significant NES as if it were a real hit

draw_df = plot_df[plot_df["FDR q-val"] < FDR_CUTOFF].copy()
 
fig, ax = plt.subplots(figsize=(1.2 * len(celltype_order) + 3, 0.35 * len(term_order) + 2))
 
sizes = draw_df["n_lead_genes"] * 8  # scale factor - adjust if dots too big/small
sc = ax.scatter(
    draw_df["celltype"].cat.codes, draw_df["Term"].cat.codes,
    s=sizes, c=draw_df["NES"], cmap=cmap, norm=norm,
    edgecolors="black", linewidths=0.4, 
)
 
ax.set_xticks(range(len(celltype_order)))
ax.set_xticklabels(celltype_order, rotation=45, ha="right")
ax.set_yticks(range(len(term_order)))
ax.set_yticklabels(term_order[::-1])
ax.set_xlabel("Cell type")
ax.set_title("Top GSEA pathways per cell type (GO Biological Process)\nHealthy vs GBM, macrophage/monocyte compartment")
 
# colorbar for NES
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("NES  (red = up in GBM, green = up in Healthy)")
 
# size legend - a few reference dot sizes for leading-edge gene count
for n in [10, 30, 60]:
    ax.scatter([], [], s=n * 8, c="grey", edgecolors="black", linewidths=0.4, label=f"{n} genes")
ax.legend(
    title="Leading-edge\ngene count", loc="upper left", bbox_to_anchor=(1.25, 1),
    frameon=False, labelspacing=1.5,
)
 
fig.tight_layout()
out_pdf = os.path.join(OUT_DIR, "10_07_gsea_dotplot_percelltype.pdf")
fig.savefig(out_pdf, bbox_inches="tight")
plt.close(fig)
print(f"Saved dotplot to {out_pdf}")

/tmp/ipykernel_425794/2585220088.py:39: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


Saved dotplot to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/10_integration_GBM_HIHA/10_07_gsea_dotplot_percelltype.pdf
